# SPY capture: a first look at the database

This notebook inventories the collected data and walks through representative records. It does **not** yet certify the collection against the BSM handoff or validate the solver.

Before running:
- Stop the collector. All connections here are **read-only** and close after each query.
- Select the standard-Python **`.venv-collector`** kernel, not the free-threaded demo environment.
- To retain results privately, save a working copy under `Tastyware_Demo_Scripts/exports/notebooks/` before running. That directory is Git-ignored. Keep this source notebook free of saved outputs.
- Run cells in order. Queries aggregate inside DuckDB or fetch small previews; they do not load the complete event history into pandas.

Times are displayed in UTC unless a column explicitly ends in `_et` (America/New_York).

## 1. Locate and open the capture safely

The default is the collector's existing database, not its pre-upgrade backup. You can replace `DATABASE_PATH` with another explicit path. A missing file raises an error rather than creating an empty database. This first version expects collector schema 2, which can also contain earlier schema-1 runs.

In [ ]:
from pathlib import Path
import json
import sys

import duckdb
import pandas as pd
from IPython.display import display

repo_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "Tastyware_Demo_Scripts" / "collector_core.py").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run this notebook from inside the Tasty_Dearcyfi_Tests repository.")

DATABASE_PATH = repo_root / "Tastyware_Demo_Scripts" / "exports" / "spy_validation.duckdb"
if not DATABASE_PATH.is_file():
    raise FileNotFoundError(f"Capture database does not exist: {DATABASE_PATH}")

def query(sql: str, parameters: list[object] | None = None) -> pd.DataFrame:
    with duckdb.connect(str(DATABASE_PATH), read_only=True) as connection:
        connection.execute("SET TimeZone='UTC'")
        return connection.execute(sql, parameters if parameters is not None else []).df()

schema = query("SELECT version FROM collector_schema")
if schema["version"].tolist() != [2]:
    raise ValueError(f"Expected collector schema 2; found {schema['version'].tolist()}.")

pd.set_option("display.max_columns", 24)
pd.set_option("display.max_colwidth", 100)
wal_path = Path(str(DATABASE_PATH) + ".wal")
print(f"Python: {sys.executable}")
print(f"DuckDB: {duckdb.__version__}; collector schema: 2")
print(f"Database: {DATABASE_PATH}")
print(f"Database size: {DATABASE_PATH.stat().st_size / 1_000_000_000:.3f} decimal GB")
print(f"WAL present: {wal_path.exists()}")
print("Read-only access; every query closes its connection.")

## 2. What tables and views exist?

Counts below are exact for base tables. Views are listed without counting them again because they derive from the same underlying records. **Do not add all table counts together as a count of unique market events.**

| Layer | Main relations | Meaning |
|---|---|---|
| Provenance | `runs`, `instruments`, `universe_revisions` | Execution history, contract terms, and subscription selections |
| Received updates | `raw_updates`, `raw_update_payloads`, `raw_records` | One delivery row per received update; the payload view joins content-addressed JSON |
| Normalized observations | `quotes`, `trades`, `analytics`, `candles` | Numeric fields and timestamps; schema-1 history was sampled, schema-2 history records received updates |
| Minute sampling | `cycles`, `samples` | Scheduled sampling attempts and references to selected observations; not time-window averages |
| Operational context | `requests`, `diagnostics`, `run_coverage` | Requests, gaps, warnings, and summary counts |

`latest_candles` is a latest-revision view, not a promise of provider finality.

In [ ]:
relations = query("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema='main' AND table_catalog=current_database()
    ORDER BY table_name
""")
row_counts = []
for relation in relations.itertuples(index=False):
    if relation.table_type == "BASE TABLE":
        identifier = '"' + relation.table_name.replace('"', '""') + '"'
        row_counts.append(int(query(f"SELECT count(*) AS rows FROM {identifier}")["rows"].iloc[0]))
    else:
        row_counts.append(pd.NA)
relations["base_table_rows"] = pd.array(row_counts, dtype="Int64")
display(relations)

### Inspect the columns of one relation

Change `RELATION_TO_INSPECT` to a name from the inventory, such as `samples`, `quotes`, or `analytics`. `record_id` identifies a row; `raw_record_id` links to retained JSON. A received update's `observation_id` links to its normalized observation.

In [ ]:
RELATION_TO_INSPECT = "raw_updates"
if RELATION_TO_INSPECT not in set(relations["table_name"]):
    raise ValueError(f"Unknown relation: {RELATION_TO_INSPECT}")
identifier = '"' + RELATION_TO_INSPECT.replace('"', '""') + '"'
display(query(f"DESCRIBE {identifier}"))

## 3. Which collection runs are present?

A restart appends a new run; it does not overwrite previous runs. This database may include failed startup attempts, short diagnostic captures, the earlier minute-only collector, and the longer unattended capture.

The recording mode is taken from each run's saved capabilities. Schema-1 runs do not have complete raw-update history, even after the database is upgraded. `partial` is a coverage/status result, not proof that the captured rows are unusable; `completed` would not certify BSM suitability either.

In [ ]:
runs = query("""
    SELECT r.run_id, r.schema_version, r.session_date,
           r.started_at, r.ended_at, r.status, r.stop_reason,
           json_extract_string(r.capabilities, '$.recording_mode') AS recording_mode,
           coalesce(u.received_updates, 0) AS received_updates
    FROM runs r
    LEFT JOIN (
        SELECT run_id, count(*) AS received_updates FROM raw_updates GROUP BY run_id
    ) u USING (run_id)
    ORDER BY received_updates DESC, r.started_at DESC
""")
if runs.empty:
    raise ValueError("The database contains no collection runs.")
display(runs)

### Choose a run for the remaining cells

By default, select the run with the largest received-update count. Replace the assignment with a specific run ID to examine another attempt. The rest of the notebook consistently filters to this run.

In [ ]:
RUN_ID = str(runs.iloc[0]["run_id"])
if RUN_ID not in set(runs["run_id"]):
    raise ValueError(f"Run is not present in this database: {RUN_ID}")
print(f"Selected run: {RUN_ID}")
display(query("SELECT * FROM run_coverage WHERE run_id=?", [RUN_ID]))

## 4. What arrived, and when?

This is a **receipt-history overview**, not a freshness or synchronization test. `source_time` is the normalized provider time and may be unknown; `received_at_utc` is local receipt/processing time. An active overall feed does not prove that every individual contract updated recently.

In [ ]:
event_summary = query("""
    SELECT kind, count(*) AS received_updates, count(DISTINCT symbol) AS symbols,
           min(received_at_utc) AS first_received_utc,
           max(received_at_utc) AS last_received_utc,
           count(*) FILTER (WHERE source_time IS NULL) AS unknown_source_time,
           count(DISTINCT connection_id) AS connections
    FROM raw_updates WHERE run_id=? GROUP BY kind ORDER BY kind
""", [RUN_ID])
for column in ("first_received_utc", "last_received_utc"):
    event_summary[column.replace("_utc", "_et")] = event_summary[column].dt.tz_convert("America/New_York")
display(event_summary)

## 5. Which instruments were selected?

Selections can be refreshed during a run. Deduplicate their reference rows by streamer symbol before counting contracts. These are the selected pilot instruments, **not all listed SPY options**. Selecting a contract does not by itself prove that it received every event type.

In [ ]:
instruments = query("""
    SELECT symbol, streamer_symbol, option_type, strike, expiration,
           exercise_style, settlement_type, shares_per_contract, stops_trading_at
    FROM instruments WHERE run_id=?
    QUALIFY row_number() OVER (
        PARTITION BY streamer_symbol ORDER BY effective_at DESC, received_at_utc DESC, record_id DESC
    )=1
    ORDER BY expiration NULLS FIRST, option_type, strike
""", [RUN_ID])
print(f"Selected instruments: {len(instruments)} (including the underlying)")
display(instruments.head(20))
display(instruments.loc[instruments["option_type"].notna()].groupby(
    ["expiration", "option_type"], dropna=False
).size().rename("contracts").reset_index())

## 6. Inspect individual quote updates

These are the last ten delivered quote updates in the selected run, not averaged prices and not necessarily ten different symbols. Normalized invalid/non-finite prices may be null; the original representation is retained in the raw payload.

`receive_sequence` orders deliveries within the run. `received_monotonic_ns` measures local timing without wall-clock adjustments; do not compare its epoch across processes. `connection_id` distinguishes reconnects.

In [ ]:
quote_preview = query("""
    SELECT u.receive_sequence, u.connection_id, u.received_at_utc,
           u.received_monotonic_ns, q.symbol,
           q.bid, q.ask, q.bid_size, q.ask_size,
           q.bid_time, q.ask_time, q.source_time, q.record_id AS quote_id
    FROM raw_updates u JOIN quotes q ON q.record_id=u.observation_id
    WHERE u.run_id=? AND u.kind='Quote'
    ORDER BY u.receive_sequence DESC LIMIT 10
""", [RUN_ID])
display(quote_preview)

### Follow one update to its original decoded payload

Change `EVENT_KIND` (`Quote`, `Trade`, `Greeks`, or `Candle`) and `EVENT_SYMBOL` to inspect another example. Candle symbols include their interval/session suffix; choose an exact symbol from the received data when inspecting candles. These payloads contain requested market-data fields, not authentication frames or original WebSocket wire bytes.

In [ ]:
EVENT_KIND = "Quote"
EVENT_SYMBOL = "SPY"
raw_example = query("""
    SELECT record_id, observation_id, kind, symbol, receive_sequence,
           received_at_utc, source_time, raw_record_id, payload
    FROM raw_update_payloads
    WHERE run_id=? AND kind=? AND symbol=?
    ORDER BY receive_sequence LIMIT 1
""", [RUN_ID, EVENT_KIND, EVENT_SYMBOL])
if raw_example.empty:
    print(f"No {EVENT_KIND} update for {EVENT_SYMBOL} in the selected run.")
else:
    display(raw_example.drop(columns="payload"))
    print(json.dumps(json.loads(raw_example.iloc[0]["payload"]), indent=2))

## 7. Compare the minute-sampling layer

A cycle is a scheduled sampling attempt; a sample is an option row from an attempt. `missed` cycles record unavailable earlier intervals rather than fabricated historical quotes. A sampled cycle can still be late or have missing inputs.

The existing sampler requires source-time alignment. Missing provider quote timestamps can leave `underlying_quote_id` and `underlying_mid` empty and `quote_eligible` false, even though SPY updates exist in the raw layer. Receipt-time reconstruction will be a separate, explicitly labeled later analysis.

In [ ]:
display(query("""
    SELECT status, count(*) AS cycles, min(scheduled_at) AS first_scheduled_utc,
           max(scheduled_at) AS last_scheduled_utc, max(lateness_seconds) AS max_lateness_seconds
    FROM cycles WHERE run_id=? GROUP BY status ORDER BY status
""", [RUN_ID]))
sample_preview = query("""
    SELECT scheduled_at, sampled_at, symbol, option_mid, underlying_mid,
           quote_eligible, analytics_fresh, quote_age_seconds,
           analytics_age_seconds, source_skew_seconds,
           quote_id, underlying_quote_id, analytics_id, quality
    FROM samples WHERE run_id=?
    ORDER BY scheduled_at DESC, symbol LIMIT 10
""", [RUN_ID])
display(sample_preview)
if not sample_preview.empty:
    print("Quality flags for the first preview row:")
    display(pd.DataFrame(json.loads(sample_preview.iloc[0]["quality"])))

## 8. Broker analytics and candle examples

Greeks and IV are broker/provider model outputs, not solver ground truth. Their conventions, rates and dividend assumptions are not certified. `analytics.price` is not relabeled as a broker mark.

Candles are provider aggregates, separate from quote events. `latest_candles` chooses a received revision; `elapsed_with_grace` is not provider finality. A later historical candle retrieval does not establish historical point-in-time availability.

In [ ]:
display(query("""
    SELECT symbol, received_at_utc, source_time, price, volatility,
           delta, gamma, theta, vega, rho, event_flags
    FROM analytics WHERE run_id=?
    ORDER BY received_at_utc DESC, record_id LIMIT 5
""", [RUN_ID]))
display(query("""
    SELECT symbol, interval_start, interval_end, received_at_utc,
           open, high, low, close, volume, count,
           elapsed_with_grace, quality
    FROM latest_candles WHERE run_id=?
    ORDER BY interval_start DESC, symbol LIMIT 5
""", [RUN_ID]))

## 9. What operational context was recorded?

These summaries help identify topics for the next investigation. Request success means a request returned or was sent, not that every subscribed symbol delivered data. Diagnostic counts are not automatically counts of bad quotes; some codes describe normal counters or acknowledged limitations.

In [ ]:
display(query("""
    SELECT code, count(*) AS records, min(received_at_utc) AS first_recorded_utc,
           max(received_at_utc) AS last_recorded_utc
    FROM diagnostics WHERE run_id=? GROUP BY code ORDER BY records DESC, code
""", [RUN_ID]))
display(query("""
    SELECT operation, status, count(*) AS requests,
           min(started_at) AS first_started_utc, max(ended_at) AS last_ended_utc
    FROM requests WHERE run_id=? GROUP BY operation, status ORDER BY operation, status
""", [RUN_ID]))

## 10. Quote resolution per minute

### Why investigate receipt-time resolution?

**Observed on October 30, 2026:** the main SPY capture has unknown quote source times, and the small SPY and actively updating Bitcoin/Ethereum quote demos print `event_time=0`, `bid_time=0`, and `ask_time=0`. Zero is treated as an unknown timestamp, not a valid market-event time. The collector still records local UTC receipt times and monotonic receive timing.

**Existing upstream reports:** [tastyware/tastytrade#195: Streaming quotes returning neither ask_time nor bid_time](https://github.com/tastyware/tastytrade/issues/195) describes the same zero bid/ask times. @Graeme22, the SDK maintainer, [responded](https://github.com/tastyware/tastytrade/issues/195#issuecomment-2585349870) that this is what the raw API data looks like and suggested a possible limitation of Tastytrade's partnership with dxFeed. The earlier [tastyware/tastytrade#119: Missing bidTime and askTime in EventType.QUOTE](https://github.com/tastyware/tastytrade/issues/119) also reproduced missing DXLink timestamps after an SDK upgrade. Both issues are closed, but neither documents a demonstrated fix for the missing bid/ask times.

**Working hypothesis, not confirmed cause:** timestamp availability is probably an upstream feed/service limitation related to the dxFeed-Tastytrade broker agreement, rather than a bug in our collector. The maintainer's suggestion about licensing/payment is speculation; we have not confirmed the agreement's terms or the cause with Tastytrade support. The reports directly address bid/ask times and do not establish the semantics of every other zero field. We are not attempting to bypass feed restrictions.

**Implication for BSM validation:** without source bid/ask times, we cannot prove exchange-time freshness, transport latency, or option/SPY synchronization. We will investigate whether frequent local receipts and sufficiently stable prices over short intervals can support explicitly approximate, receipt-time-based comparisons. This section starts with quote counts per minute; within-minute arrival gaps, bid/ask/midpoint variability, and option/SPY pairing remain separate checks. Frequent updates alone do not certify the data or replace the missing source timestamps.

### Counting method

Count **received Quote deliveries**, including repeated payloads and invalid prices, not distinct prices or exchange ticks. Minute buckets use local UTC receipt time, with New York labels for inspection. Only the selected run and its SPY options are included; SPY itself is excluded.

Build every minute from each option's first recorded selection through the earlier of run end, regular-session close, or contract trading stop. Selection time is a proxy for intended subscription exposure, not proof that subscription was active. Setup delays and reconnect gaps remain in the denominator; minutes before selection do not. A zero-update minute may mean an unchanged market or a delivery gap.

Statistics use **complete 60-second minutes only**, including zeros. Partial boundary minutes and their deliveries are retained separately, not scaled up. Schema-1 runs cannot support this analysis. Counts alone do not establish uniform arrival spacing, low price variance, source freshness, or BSM suitability; those are later checks.

In [ ]:
recording_mode = query("SELECT json_extract_string(capabilities, '$.recording_mode') AS mode FROM runs WHERE run_id=?", [RUN_ID])
if recording_mode['mode'].tolist() != ['all_received_updates_and_minute_samples']:
    raise ValueError('Select a run that retained all received updates; minute-only runs cannot measure delivery frequency.')

quote_minute_counts = query("""
    WITH options AS (
        SELECT streamer_symbol, min(effective_at) AS first_selected_at,
               min(stops_trading_at) AS stops_trading_at,
               min(expiration) AS expiration, min(option_type) AS option_type,
               min(strike) AS strike
        FROM instruments WHERE run_id=? AND option_type IS NOT NULL
          AND streamer_symbol LIKE '.SPY%'
        GROUP BY streamer_symbol
    ), bounds AS (
        SELECT o.*, greatest(r.started_at, r.session_open, o.first_selected_at) AS exposure_start,
               least(r.ended_at, r.session_close, o.stops_trading_at) AS exposure_end
        FROM options o CROSS JOIN runs r WHERE r.run_id=? AND r.ended_at IS NOT NULL
    ), grid AS (
        SELECT b.*, m.minute_utc,
               greatest(b.exposure_start, m.minute_utc) AS interval_start,
               least(b.exposure_end, m.minute_utc + INTERVAL '1 minute') AS interval_end
        FROM bounds b CROSS JOIN LATERAL generate_series(
            date_trunc('minute', b.exposure_start), date_trunc('minute', b.exposure_end),
            INTERVAL '1 minute'
        ) m(minute_utc)
        WHERE b.exposure_end > b.exposure_start AND m.minute_utc < b.exposure_end
    )
    SELECT g.streamer_symbol, g.expiration, g.option_type, g.strike,
           g.first_selected_at, g.minute_utc,
           epoch(g.interval_end - g.interval_start) AS exposure_seconds,
           g.interval_end - g.interval_start = INTERVAL '1 minute' AS full_minute,
           count(u.record_id) AS quote_count
    FROM grid g LEFT JOIN raw_updates u
      ON u.run_id=? AND u.kind='Quote' AND u.symbol=g.streamer_symbol
     AND u.received_at_utc >= g.interval_start AND u.received_at_utc < g.interval_end
    GROUP BY ALL ORDER BY g.streamer_symbol, g.minute_utc
""", [RUN_ID, RUN_ID, RUN_ID])
if quote_minute_counts.empty:
    raise ValueError('No selected SPY-option exposure intervals found in this completed run.')
quote_minute_counts['minute_et'] = quote_minute_counts['minute_utc'].dt.tz_convert('America/New_York')
full_quote_minutes = quote_minute_counts.loc[quote_minute_counts['full_minute']].copy()

option_quote_count_stats = full_quote_minutes.groupby('streamer_symbol')['quote_count'].agg(
    full_minutes='size', full_minute_quotes='sum', zero_quote_minutes=lambda x: int(x.eq(0).sum()),
    min_per_minute='min', p05_per_minute=lambda x: x.quantile(0.05),
    median_per_minute='median', mean_per_minute='mean', std_per_minute='std',
    p95_per_minute=lambda x: x.quantile(0.95), max_per_minute='max',
).reset_index()
exposure_summary = quote_minute_counts.groupby('streamer_symbol').agg(
    expiration=('expiration', 'first'), option_type=('option_type', 'first'), strike=('strike', 'first'),
    first_selected_at=('first_selected_at', 'first'), exposed_minutes=('quote_count', 'size'),
    total_quotes=('quote_count', 'sum'),
).reset_index()
partial_summary = quote_minute_counts.loc[~quote_minute_counts['full_minute']].groupby('streamer_symbol').agg(
    partial_minutes=('quote_count', 'size'), partial_minute_quotes=('quote_count', 'sum'),
).reset_index()
option_quote_count_stats = exposure_summary.merge(option_quote_count_stats, on='streamer_symbol', how='left').merge(
    partial_summary, on='streamer_symbol', how='left'
)
count_columns = ['full_minutes', 'full_minute_quotes', 'zero_quote_minutes', 'partial_minutes', 'partial_minute_quotes']
option_quote_count_stats[count_columns] = option_quote_count_stats[count_columns].fillna(0).astype('int64')
denominator = option_quote_count_stats['full_minutes'].replace(0, float('nan'))
option_quote_count_stats['zero_quote_pct'] = 100 * option_quote_count_stats['zero_quote_minutes'] / denominator
option_quote_count_stats = option_quote_count_stats.sort_values(['expiration', 'option_type', 'strike']).reset_index(drop=True)
assert (option_quote_count_stats['total_quotes'] == option_quote_count_stats['full_minute_quotes'] + option_quote_count_stats['partial_minute_quotes']).all()
print(f"Options: {len(option_quote_count_stats)}; option-minute buckets: {len(quote_minute_counts):,}")
print(f"Received option quotes in these exposure intervals: {quote_minute_counts['quote_count'].sum():,}")
print('std_per_minute is sample standard deviation of counts, not price variance; undefined with fewer than two full minutes.')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    formatted_stats = option_quote_count_stats.copy()
    numeric_columns = formatted_stats.select_dtypes(include='number').columns
    formatted_stats[numeric_columns] = formatted_stats[numeric_columns].round(2)
    display(formatted_stats)


In [ ]:
# Change this symbol to inspect its minute-by-minute counts, including zeros.
OPTION_TO_INSPECT = str(option_quote_count_stats.iloc[0]['streamer_symbol'])
if OPTION_TO_INSPECT not in set(option_quote_count_stats['streamer_symbol']):
    raise ValueError(f'Option is not present in the selected run: {OPTION_TO_INSPECT}')
option_minute_detail = quote_minute_counts.loc[
    quote_minute_counts['streamer_symbol'].eq(OPTION_TO_INSPECT),
    ['minute_et', 'quote_count', 'exposure_seconds', 'full_minute'],
]
print(f'{OPTION_TO_INSPECT}: first 20 minute buckets; full data remain in option_minute_detail')
display(option_minute_detail.head(20))
print('Distribution of counts across all complete option-minute buckets (not an equally weighted contract average):')
display(full_quote_minutes['quote_count'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_frame('quotes_per_option_minute'))

## Next: turn exploration into handoff checks

This first pass establishes **what is present**, not whether it is sufficient for BSM validation. We can incrementally add:

1. Per-instrument coverage, update frequencies, sequence integrity and reconnect gaps.
2. Missingness and quote quality distributions; explicit receipt-time option/SPY alignment experiments.
3. Mapping of contract terms, IV/Greek units, candle revisions and provenance to each handoff requirement.
4. The external rate/dividend/model assumptions needed before numerical BSM comparisons.

References:
- [Collector operator and schema guide](https://github.com/khazaddim/Tasty_Dearcyfi_Tests/blob/main/Tastyware_Demo_Scripts/README.md)
- [BSM data-collection handoff](https://github.com/khazaddim/Macro_Ideas/blob/main/short_put_replay/quote_collector_requirements.md)
- [BSM implementation](https://github.com/khazaddim/Macro_Ideas/blob/main/short_put_replay/bsm_model.py)

No API credentials or network requests are needed by this notebook. Preserve the database and WAL together if moving a stopped capture, and do not commit captured data or saved private notebook outputs.